<a href="https://colab.research.google.com/github/23071121-pixel/NNDL_Practicals/blob/main/NNDL_Practical_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

# 1. Set Parameters
VOCAB_SIZE = 10000
MAX_LENGTH = 200
EMBEDDING_DIM = 128

# 2. Load IMDB Dataset
print("Loading IMDB dataset...")
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)
print("\nDataset loaded successfully!")
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

# 3. Pad Sequences
print("\nPadding sequences...")
X_train = pad_sequences(
    X_train,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)
X_test = pad_sequences(
    X_test,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)
print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

# 4. Build RNN Model using LSTM
model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM),
    LSTM(64, dropout=0.2, recurrent_dropout=0.2),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

# 5. Compile Model
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# 6. Display Model
print("\nModel Summary:")
model.summary()

# 7. Early Stopping
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

# 8. Train Model
print("\nTraining model...")
history = model.fit(
    X_train,
    y_train,
    epochs=8,
    batch_size=128,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

# 9. Evaluate Model
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)
print("\nTest Loss:", round(test_loss, 4))
print("Test Accuracy:", round(test_accuracy * 100, 2), "%")

# 10. Get Word Index
word_index = imdb.get_word_index()

# 11. Encode User Text
def encode_text(text):
    words = text.lower().split()
    encoded = []
    for word in words:
        word = word.strip(".,!?;:\"'()[]{}")
        if word in word_index:
            index = word_index[word] + 3
            if index < VOCAB_SIZE:
                encoded.append(index)
            else:
                encoded.append(2)
        else:
            encoded.append(2)
    return encoded

# 12. Predict Sentiment
def predict_sentiment(text):
    encoded_text = encode_text(text)
    padded_text = pad_sequences(
        [encoded_text],
        maxlen=MAX_LENGTH,
        padding="post",
        truncating="post"
    )
    prediction = model.predict(padded_text, verbose=0)[0][0]
    if prediction >= 0.5:
        sentiment = "Positive"
        confidence = prediction * 100
    else:
        sentiment = "Negative"
        confidence = (1 - prediction) * 100
    return sentiment, confidence

# 13. Test Sample Sentences
test_sentences = [
    "I absolutely loved this movie. It was fantastic and entertaining.",
    "This movie was amazing and the acting was excellent.",
    "I really enjoyed this film and would recommend it to everyone.",
    "The story was beautiful and the characters were wonderful.",
    "This was one of the best movies I have ever watched.",
    "The movie was terrible and extremely boring.",
    "I hated this film because the acting was awful.",
    "This was a complete waste of time and money.",
    "The story was boring and the characters were disappointing.",
    "I would never recommend this horrible movie."
]

print("\nSentiment Predictions:")
for sentence in test_sentences:
    sentiment, confidence = predict_sentiment(sentence)
    print("\nSentence:", sentence)
    print("Sentiment:", sentiment)
    print("Confidence:", round(confidence, 2), "%")

# 14. Interactive Sentiment Analysis
print("\nCustom Sentiment Analysis")
print("Type 'exit' to stop the program.")
while True:
    user_text = input("\nEnter a sentence: ")
    if user_text.lower() == "exit":
        print("Program ended.")
        break
    if user_text.strip() == "":
        print("Please enter a valid sentence.")
        continue
    sentiment, confidence = predict_sentiment(user_text)
    print("Predicted Sentiment:", sentiment)
    print("Confidence:", round(confidence, 2), "%")

Loading IMDB dataset...
17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

Dataset loaded successfully!
Training samples: 25000
Testing samples: 25000

Padding sequences...
Training data shape: (25000, 200)
Testing data shape: (25000, 200)

Model Summary:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Training model...
Epoch 1/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 75s 455ms/step - accuracy: 0.5253 - loss: 0.6918 - val_accuracy: 0.5066 - val_loss: 0.6955
Epoch 2/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 69s 437ms/step - accuracy: 0.5876 - loss: 0.6647 - val_accuracy: 0.6220 - val_loss: 0.6415
Epoch 3/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 81s 433ms/step - accuracy: 0.6865 - loss: 0.6075 - val_accuracy: 0.7102 - val_loss: 0.5828
Epoch 4/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 70s 446ms/step - accuracy: 0.7680 - loss: 0.5246 - val_accuracy: 0.7492 - val_loss: 0.5567
Epoch 5/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 71s 453ms/step - accuracy: 0.7628 - loss: 0.5039 - val_accuracy: 0.7516 - val_loss: 0.6029
Epoch 6/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 69s 442ms/step - accuracy: 0.7839 - loss: 0.4856 - val_accuracy: 0.7652 - val_loss: 0.5491
Epoch 7/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 83s 452ms/step - accuracy: 0.7883 - loss: 0.4825 - val_accuracy: 0.6804 - val_loss: 0.6787
Epoch 8/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 81s 443ms/step - accuracy: 0.823